# Layer management with `LayerStack`

A walk-through of the layer-centric workflow for building MODFLOW 6 discretization
on a Voronoi grid:

1. build a grid
2. make some source rasters
3. declare a **layer stack** (top + named layers, each with its own bottom and policy)
4. area-weighted sampling, per-layer pinch-out, units, preview
5. surface algebra, provenance-aware caching
6. hand the result to `mf.disv`

`LayerStack` is a thin facade over `LayerSurfaces` -- it adds names, per-layer
config, and edit-by-name ergonomics, but reuses the same sampling/reconcile/pinch
engine.

In [ ]:
from pathlib import Path

import numpy as np
import rasterio
from rasterio.transform import from_origin
import matplotlib.pyplot as plt

import myflopy as mf
from myflopy.layers import LayerStack, Raster, Flat, Contours, Isopach, Min, Max, Clamp, Where
from myflopy.surfaces import Surface

## 1. A small Voronoi grid

In [ ]:
workspace = Path.cwd() / "artifacts" / "layer_management"
workspace.mkdir(parents=True, exist_ok=True)

tri = mf.TriangleGrid(model_ws=str(workspace), angle=30)
tri.set_domain_rectangle(x_dist=1000, y_dist=600, origin=(0, 0))
tri.add_region_rectangle(
    origin=(300, 150), x_dist=400, y_dist=300, max_area=2500, label="refine", priority=1
)
tri.build()
vor = mf.VoronoiGridPlus(tri)
print(f"grid: {vor.ncpl} cells, CRS = {vor.crs}")

## 2. Synthetic source rasters

A ground DEM and a bedrock surface (with a nodata hole, to show `fill="propagate"`),
plus the same ground surface expressed in **metres** to demonstrate unit conversion.

In [ ]:
def write_raster(path, func, *, nodata=-9999.0, px=10.0, pad=50.0,
                 crs="EPSG:2927", hole=None):
    """Write a north-up GeoTIFF covering the model domain in the grid CRS."""
    xs = np.arange(-pad, 1000 + pad, px)
    ys = np.arange(600 + pad, -pad, -px)
    X, Y = np.meshgrid(xs, ys)
    Z = func(X, Y).astype("float64")
    if hole is not None:
        cx, cy, r = hole
        Z[(X - cx) ** 2 + (Y - cy) ** 2 < r ** 2] = nodata
    transform = from_origin(xs[0] - px / 2, ys[0] + px / 2, px, px)
    with rasterio.open(
        path, "w", driver="GTiff", height=Z.shape[0], width=Z.shape[1], count=1,
        dtype="float64", crs=crs, transform=transform, nodata=nodata,
    ) as dst:
        dst.write(Z, 1)
    return path


ground = write_raster(workspace / "ground.tif", lambda x, y: 100 - 0.02 * x - 0.01 * y)
bedrock = write_raster(
    workspace / "bedrock.tif", lambda x, y: 50 - 0.03 * x, hole=(500, 300, 120)
)
ground_m = write_raster(
    workspace / "ground_m.tif", lambda x, y: (100 - 0.02 * x - 0.01 * y) / 3.280839895
)
print("wrote", ground.name, bedrock.name, ground_m.name)

## 3. Declare the layer stack

Each `add(...)` is a named layer. A layer's bottom can be an absolute surface
(`bottom=Raster(...)`) or a thickness below the layer above (`thickness=...`, a
scalar or an `Isopach`). Per-layer `min_thickness` / `pinch` override the build
defaults.

In [ ]:
stack = (
    LayerStack(vor, top=Raster(ground), length_units="feet")   # area-weighted sampling, feet/days
    .add("alluvium", thickness=20, min_thickness=2)
    .add("clay",     thickness=Isopach(Flat(15)))              # uniform 15-ft isopach
    .add("sand",     thickness=30, pinch="passthrough")
    .add("bedrock",  bottom=Raster(bedrock, fill="propagate")) # nodata hole -> drape from above
)
result = stack.build()
print(result.report())

## 4. Pinch-out and idomain

Cells thinner than their layer's `min_thickness` become MODFLOW pass-through
(`idomain = -1`) so vertical flow still connects across the pinch. `inactive`
(`0`) and `floor` (stay active) are the other policies.

In [ ]:
vals, counts = np.unique(result.idomain, return_counts=True)
print("idomain values -> counts:", dict(zip(vals.tolist(), counts.tolist())))
print("pass-through (pinched) cells:", result.n_pinched)
print("top/botm/idomain shapes:", result.top.shape, result.botm.shape, result.idomain.shape)

## 5. Preview the thickness

In [ ]:
ax = stack.plot.map().axes      # total thickness choropleth
ax.figure.set_size_inches(6, 4)
plt.show()

In [ ]:
ax = stack.plot.map(layer="clay").axes
ax.figure.set_size_inches(6, 4)
plt.show()

## 6. Units

Declare a source's `units=` and the stack converts it to the model length unit.
Here the *same* ground surface, stored in metres, lands on the same feet elevations.

In [ ]:
ft = LayerStack(vor, top=Raster(ground), length_units="feet").add("z", bottom=Flat(0)).build()
m = LayerStack(vor, top=Raster(ground_m, units="meters"), length_units="feet").add("z", bottom=Flat(0)).build()
print("feet source  top[:3]:", np.round(ft.top[:3], 4))
print("metre source top[:3]:", np.round(m.top[:3], 4))
print("max abs difference:", float(np.nanmax(np.abs(ft.top - m.top))))

## 7. Surface algebra

Surfaces compose with fluent methods that read base-surface-first, left to right:
`.capped_at(s)` (never above), `.floored_at(s)` (never below),
`.between(lower=, upper=)`, `.within(zone, outside=)`, and vertical offsets like
`ground - 5` (or `.below(5)` / `.above(5)`). These are the explicit version of the
onlap/erosion rules geologists use. (The `Min` / `Max` / `Clamp` / `Where`
constructors still work.) Here bedrock is capped 5 ft below ground.

In [ ]:
ground_surf = Raster(ground)
# read it left to right: the bedrock raster, capped at 5 ft below ground
constrained = Raster(bedrock, fill="propagate").capped_at(ground_surf - 5)
algebra = LayerStack(vor, top=ground_surf).add("overburden", bottom=constrained)
r2 = algebra.build()
print(r2.report())

## 8. Provenance-aware caching (derived surfaces)

A `Contours` surface interpolates to a cached raster, recorded with a provenance
sidecar. If the contours change, the cache is reported **stale** and a build warns;
you rebuild explicitly with `refresh=True` (or `stack.refresh()`). This needs GRASS
to actually interpolate, so here we only inspect the status (no interpolation runs).

In [ ]:
contour = Contours(workspace / "bedrock_contours.gpkg", z="elev",
                   out=workspace / "bedrock_from_contours.tif")
demo = LayerStack(vor, top=Raster(ground)).add("bedrock", bottom=contour)
print("cache status:", demo.cache_status())   # {'bedrock': 'missing'} until first build/refresh

## 9. Hand off to DISV

In [ ]:
disv = stack.to_disv(vor, name="disv")
print("disv option keys:", sorted(disv.options))
print("length_units:", disv.options["length_units"])
print("idomain shape:", np.asarray(disv.options["idomain"]).shape)
# `disv` is a ready PackageSpec -- drop it into mf.gwf(...) / mf.simulation(...).

## 10. Vary the stack by name

In [ ]:
stack.replace("clay", thickness=Isopach(Flat(25)))   # thicker clay
stack.insert_below("clay", "silt", thickness=10)     # add a silt lens
print("layers:", stack.names)
print(stack.build().report())

stack.remove("silt")
print("layers after remove:", stack.names)

---
# 11. A stream valley eroding layered sediments

A realistic, end-to-end example that ties the pieces together:

1. survey **contours** of a stream valley → interpolate a ground surface (GRASS, cached)
2. the valley **erodes** the flat sediment layers beneath it (reconcile + pinch-out)
3. view the result as a **cross-section** and in **3D** (two ways)

### Check GRASS is available

`Surface.from_contours` runs the GRASS `r.surf.contour` interpolation (see
`modflow/utils/contour_interp.py`). GRASS is a **system** dependency (not pip);
install it with your package manager or via OSGeo4W and discovery is automatic —
the launcher is found on `PATH` (or in the OSGeo4W/QGIS bundles on Windows), and
its Python bindings, which live inside the install at `<prefix>/etc/python`, are
located by asking that launcher. Set `GRASS_BIN` only if it isn't discoverable.

This cell is a preflight check, not setup: it just reports what was found.

In [ ]:
# Preflight only -- `Surface.from_contours` does this itself. These are internals
# (there is no public discovery API); they are here to fail loudly and early
# rather than midway through section 11.
from myflopy.modflow.utils.contour_interp import _default_grass_bin, _grass_modules

print("GRASS launcher:", _default_grass_bin())
_grass_modules()
print("GRASS python ready")

### Survey contours of a stream valley

We synthesize a Gaussian valley (a stream along `y = 300`, gentle downstream slope)
and extract elevation contour lines, as if surveyed -- then GRASS interpolates them
back to a smooth ground raster.

In [ ]:
import geopandas as gpd
from shapely.geometry import LineString

px = 10.0
gx = np.arange(0, 1000 + px, px)
gy = np.arange(0, 600 + px, px)
GX, GY = np.meshgrid(gx, gy)
# ridge ~105 ft; the stream cuts a ~33 ft valley to ~70 ft along y = 300.
DEM = 105 - 33 * np.exp(-((GY - 300) / 120) ** 2) - 0.005 * GX

cs = plt.contour(GX, GY, DEM, levels=np.arange(72, 108, 2.0))
rows = [
    {"elev": float(lv), "geometry": LineString(seg)}
    for lv, segs in zip(cs.levels, cs.allsegs)
    for seg in segs if len(seg) >= 2
]
plt.close()
contours_path = workspace / "valley_contours.gpkg"
gpd.GeoDataFrame(rows, crs=str(vor.crs)).to_file(contours_path, driver="GPKG")

# Region raster defines the GRASS interpolation extent + resolution.
region_path = workspace / "region.tif"
reg = np.ones((len(gy), len(gx)))
with rasterio.open(
    region_path, "w", driver="GTiff", height=reg.shape[0], width=reg.shape[1], count=1,
    dtype="float64", crs=str(vor.crs), transform=from_origin(gx[0], gy[-1] + px, px, px),
) as dst:
    dst.write(reg, 1)
print(f"{len(rows)} contour segments; valley DEM {DEM.min():.0f}-{DEM.max():.0f} ft")



### Interpolate the ground surface (cached derived raster)

In [ ]:
ground = Surface.from_contours(
    contours_path, z="elev", region_raster=region_path,
    out=workspace / "valley_ground.tif", resolution=10, epsg="2927",
)
print("cache status:", ground.cache_status())     # 'missing'
ground_tif = ground.resolve_source(refresh=True)               # runs GRASS r.surf.contour
print("cache status:", ground.cache_status())     # 'fresh' (+ provenance sidecar written)

with rasterio.open(ground_tif) as src:
    g = src.read(1, masked=True)
plt.figure(figsize=(6, 3.6))
plt.imshow(g, extent=(0, 1000, 0, 600), origin="upper", cmap="terrain")
plt.colorbar(label="ground elevation [ft]")
plt.title("Interpolated valley ground surface (from contours)")
plt.show()

### Build the eroding stack

`top = ground` (the valley). The sediment layers are flat; where the valley cuts
below a layer, reconcile pushes that layer to ~zero thickness and pinch-out marks
it `idomain = -1`. The stream therefore **erodes** the sand and clay along its course.

In [ ]:
erosion = (
    LayerStack(vor, top=ground, length_units="feet")
    .add("sand",    bottom=Flat(85), min_thickness=2)
    .add("clay",    bottom=Flat(70), min_thickness=2)
    .add("till",    bottom=Flat(55), min_thickness=2)
    .add("bedrock", bottom=Flat(35))
)
eroded = erosion.build()
print(eroded.report())

## The views

Every picture comes from the same four verbs on `result.plot`: `map()` for the
**thickness**, `section()` for a **cross-section** (pass `x=`/`y=`/`line=` to aim it
across the valley), `surface()` for a **3D contact**, and `grid()` for the
**3D layered grid** (VTK, colored by layer).

Each returns a *Picture*: it renders itself, and answers `.show()`, `.save(path)`
and `.html(path)` -- so nothing is written to disk unless you ask for it.

In [ ]:
# the same four views `views(x=500)` used to render in one call
from IPython.display import display

eroded.plot.map().show()
eroded.plot.section(x=500).show()
display(eroded.plot.surface("all"))
display(eroded.plot.grid())

## Choose which surface(s) to plot in 3D

`plot.surface()` defaults to the model **top**, but any contact is plottable. The
choices are `result.surface_names` (`"top"` plus each layer's bottom). Pass one
name, a **list** of names, or `"all"` to overlay them in a single scene. A single
surface with relief is shaded by elevation; a flat surface or several surfaces get
distinct solid colors (use `color_by="elevation"` to force the shared ramp).

To save one, call `.html(path)` on the picture -- the same call on every picture,
rather than an `html_path=` argument on this one.

In [ ]:
print("surfaces:", eroded.surface_names)

# a single layer -- even a flat one (here the bedrock bottom) now renders reliably
eroded.plot.surface("clay")

In [ ]:
# overlay several contacts -- the valley top vs. the sand and clay bottoms
eroded.plot.surface(["top", "sand", "clay"])

## Show only some layers in 3D (VTK)

`plot.grid()` draws the whole grid by default, but the first argument focuses on a
single layer or a subset -- by name or index. Colors stay keyed to each layer's
position, so a subset keeps the same colors it has in the full stack.

`grid()` rather than `surface()` because this is the *mesh itself* in 3D, not a
height field; `backend="plotly"` gives the same mesh drawn flat.

In [ ]:
# just the two eroded layers (sand + clay); pass a single name or index too
eroded.plot.grid(["sand", "clay"])

---
## 12. QC the stack before MODFLOW runs

`stack.qc()` (or `result.qc()`) catches what makes MODFLOW 6 fail or mislead:
cells with **no source coverage** (NaN-bounded active cells), **non-positive
thickness** on active cells, and **isolated active cells** with no connection to a
neighbour. It also counts connected active components and — from `stack.qc()` —
how much **reconcile** had to move each layer. `result.validate()` raises on fatal
problems; `result.prune_isolated()` deactivates orphan cells.

In [ ]:
# reconcile_moved shows the stream eroding the sand/clay (surfaces pushed apart)
print(erosion.qc())

## 13. Start from an existing MODFLOW model

`LayerStack.from_modflow(vor, source, ...)` reads an existing model's top/botm
straight off its grid — interpolating onto your grid (`resample=True`, the default)
or using the arrays verbatim on a matching grid. The result is an ordinary
`LayerStack` you can edit (`.replace(...)`, `.add(...)`) and rebuild. Here we
round-trip the eroded model's own grid as a stand-in for one you'd load with flopy.

In [ ]:
existing = eroded.vertex_grid()      # stand-in for a model you loaded with flopy
reused = LayerStack.from_modflow(
    vor, existing, names=["sand", "clay", "till", "bedrock"]
)
print(reused.build().report())